# LogisticRegression: 머신러닝 분류(Classification) 모델 

# A) 이진분류 모델 구현 

In [33]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 

from sklearn.model_selection import train_test_split 
from sklearn.preprocessing import StandardScaler 
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import recall_score, precision_score, f1_score, classification_report

## 1. 문제 정의 

- 생선의 특성을 통해 어떤 생선으로 분류되는지 예측하는 이진분류 모델을 구현해보자 

## 2. 데이터 로드 및 탐색 

- 데이터: 생선 데이터
- 포맷: csv 
- feature(특성):
    - Weight: 무게
    - Length: 길이
    - Diagonal: 대각선 길이
    - Height: 높이 
    - width: 두께
- target:
    - Species: 종 (총 2종, Bream, Smelt)


In [3]:
df = pd.read_csv("../data/fish_bream_smelt.csv")

print(df.shape)
df.head()

(49, 6)


,Species,Weight,Length,Diagonal,Height,Width
0,Bream,242.0,25.4,30.0,11.5200,4.0200
1,Bream,290.0,26.3,31.2,12.4800,4.3056
2,Bream,340.0,26.5,31.1,12.3778,4.6961
3,Bream,363.0,29.0,33.5,12.7300,4.4555
4,Bream,430.0,29.0,34.0,12.4440,5.1340


In [ ]:
df.info() # 데이터 결측치와 자료형 확인 

# 결측치: 존재하지 않음
# 데이터 타입:
# - 수치형: 스케일링 필요한지 확인
# - 문자형: 인코딩(수치화) 필요한지 확인

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49 entries, 0 to 48
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Species   49 non-null     object 
 1   Weight    49 non-null     float64
 2   Length    49 non-null     float64
 3   Diagonal  49 non-null     float64
 4   Height    49 non-null     float64
 5   Width     49 non-null     float64
dtypes: float64(5), object(1)
memory usage: 2.4+ KB


In [5]:
# 결측치 확인 방법2 
df.isnull().sum()

Species     0
Weight      0
Length      0
Diagonal    0
Height      0
Width       0
dtype: int64

In [ ]:
# 통계 확인
df.describe() # 데이터마다 스케일 확인 -> 스케일링 진행 여부 결정 

,Weight,Length,Diagonal,Height,Width
count,49.000000,49.000000,49.000000,49.000000,49.000000
mean,444.500000,27.055102,31.120408,11.476400,4.259751
std,328.143233,10.242804,12.097296,6.150976,1.967686
min,6.700000,9.800000,10.800000,1.728400,1.047600
25%,19.700000,14.300000,15.200000,2.872800,1.879200
50%,500.000000,31.000000,36.200000,14.179500,5.072800
75%,700.000000,34.500000,39.700000,15.633000,5.589000
max,1000.000000,41.000000,46.500000,18.957000,6.749700


## 3. 데이터 분리 

### 3.1 features와 target으로 분리 

In [7]:
X = df.iloc[:, 1:]
y = df[["Species"]]

print(X.shape, y.shape)

(49, 5) (49, 1)


### 3.2 Train, Test 분리 

1. y값의 분포 확인 

In [ ]:
# target의 클래스 분포 확인 
y.value_counts() # 클래스 불균형이 있음을 확인 

# 클래스 불균형이 존재할 경우 
# - 모델이 특정 클래스에 대해서만 학습능력 및 성능이 좋아지기 때문에 일반화 성능이 저하됨 
# - 더 적은 범주의 데이터 양을 늘려주거나 더 많은 범주의 데이터양을 줄여줘서 비율을 맞춰줄 필요성이 있음 
# - 또한, train, test 분류 시 train_test_split 함수를 통해 랜덤하게 데이터가 분류될 경우, 특정 범주만
#   train에 뽑힐 가능성이 커짐 -> 모델이 특정 클래스에 대해서만 학습을 진행하게되므로 성능 저하 발생 
# - 따라서, 동일한 비율로 train, test 분류 진행이 필요함 

Species
Bream      35
Smelt      14
Name: count, dtype: int64

2. train, test 분리 

In [9]:
# stratify=y: 각 클래스가 차지하는 비율을 train, test에 동일하게 적용시켜줌 

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    stratify=y,
    random_state=42
)

print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

(39, 5) (39, 1) (10, 5) (10, 1)


In [12]:
# 클래스 비율 확인 
print(y_train.value_counts(normalize=True))
print()
print(y_test.value_counts(normalize=True))

Species
Bream      0.717949
Smelt      0.282051
Name: proportion, dtype: float64

Species
Bream      0.7
Smelt      0.3
Name: proportion, dtype: float64


## 4. 전처리: 스케일링 

- 수치형 데이터 스케일링: 스케일이 특성마다 너무 차이가 날 경우 값의 범위가 큰 특성이 모델의 학습에 지나치게 영향을 주고 편향 학습이 될 가능성이 큰데 이를 방지하여 모델이 공정하게 학습할 수 있도록 함 
- StandardScaler(): 평균이 0, 표준편차가 1이되도록 데이터 변환 
    - 값의 범위: 보통 -3~3사이
    - 이상치 영향: 있음(평균, 표준편차에 민감)
    - 사용모델: 선형 모델
- MinMaxScaler(): 최솟값을 0, 최댓값이 1이 되도록 데이터 변환
    - 이상치 영향: 큼 (이상치가 있을 경우 거의 0에 몰림)
    - 사용모델: 신경망, 이미지 등 

In [13]:
# 스케일러 객체 생성 
scaler = StandardScaler()

# 스케일러에 X_train의 특성들 범위 학습시키기
scaler.fit(X_train)

# 학습된 scaler를 train, test 데이터에 적용
X_train_ss = scaler.transform(X_train)
X_test_ss = scaler.transform(X_test)

In [ ]:
# ddof=0: 데이터가 모집단임을 의미
# ddof=1: 데이터가 표본임을 의미 
print(np.std(X_train_ss, axis=0, ddof=1)) # 각 칼럼의 표준편차 확인 
print(np.mean(X_train_ss, axis=0)) # 각 칼럼의 평균 확인

[1.01307245 1.01307245 1.01307245 1.01307245 1.01307245]
[-5.40877884e-17 -3.70074342e-16  1.59416639e-16 -2.93212748e-16
  3.70074342e-16]


## 5. 모델링 및 정확도 확인 

- LogisticRegression()의 매개변수 
    - C: LogisticRegression에서 규제를 제어하는 매개변수
        - LogisticRegression은 기본적으로 릿지 회귀 처럼 계수의 제곱을 규제 (L2규제)
        - C는 Ridge회귀의 alpha로 값이 작을수록 규제가 커짐
        - 기본 값은 1 

In [15]:
# 모델 객체 생성 
model = LogisticRegression()

# 모델 학습 
model.fit(X_train_ss, y_train)

# 모델 성능 평가 (정확도)
print(model.score(X_train_ss, y_train))
print(model.score(X_test_ss, y_test))

1.0
1.0


/opt/anaconda3/envs/sesac-env/lib/python3.11/site-packages/sklearn/utils/validation.py:1352: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


## 6. metrics로 모델 평가 
- 정확도(accuracy):
    - 전체 예측 중 맞게 예측한 비율 (맞다고 예측하든 아니라고 예측하든 예측이 맞은 경우)
    - 데이터의 불균형 문제가 있을 경우, 90%가 양성클래스(1)인 데이터를 학습했을 때, 모델이 새로운 데이터를 모두 양성이라고 예측해도 정확도가 90%를 넘겨버림\
    이 경우 recall은 0이 됨   

- 재현율(recall): 
    - 실제 positive 중 positive로 맞게 예측한 비율
    - 즉, 실제 정답을 정답이라고 맞게 예측한것의 비율
    - ex) 실제 정답도 화이트와인인데, 화이트와인으로 맞게 예측한 것의 비율
    - 실제 문제를 놓치면 안되는 경우 유용 
        - 암 진단 시스템: 진짜 환자를 놓치면 위험하므로 recall이 중요
        - 불법 행위 탐지(부정 결제, 테러 등): 실제 문제를 놓치면 큰 사고로 이어지므로 recall을 최대한 높여야 함 

- 정밀도(precision): 
    - positive로 예측한 것 중 실제도 positive인것의 비율
    - 즉, 맞다고 예측했는데 실제도 맞는 것의 비율
    - 화이트와인일거라고 예측했는데 실제로도 화이트와인인 것의 비율
    - 오탐이 큰 피해를 주는 경우 
        - 스펨 필터링: 스펨이 아닌 정상 메일을 스펨으로 분류하면 곤란 -> 정밀도가 중요
        - 암 진단에 양성 예측 후 재검사 하는 경우: 진짜 암이 아닌데도 양성이라고 나오면 환자에게 심리적 부담 

- f1_score: 
    - 민감도와 재현율의 조화 평균
    - 0~1 사이로, 클 수록 좋으며 
    - 클래스 불균형일 때 유용
    - 재현율과 정밀도의 균형이 중요할 때 유용 

- classification_report
    - macro avg(매크로 평균):
        - 각 클래스별로 개별적으로 정밀도, 재현율, F1 점수를 계산한 뒤, 이 값들의 산술 평균을 구합니다.
        - 즉, 클래스별로 동일한 가중치(비율과 상관없이)를 부여하여 평균을 냅니다.
        - 클래스 불균형이 심할 때 소수 클래스의 성능도 동일하게 반영됩니다.

    - weighted avg(가중 평균):
        - 각 클래스별로 정밀도, 재현율, F1 점수를 계산한 뒤, 각 클래스의 샘플 개수(비율)를 가중치로 곱해서 평균을 냅니다.
        - 즉, 데이터에서 많이 등장하는 클래스가 평균에 더 큰 영향을 미칩니다.
        - 전체 데이터의 분포를 반영한 평균값입니다.

    - 정리:
        - macro avg: 클래스별 성능을 '동등하게' 평가하고 싶을 때 사용
        - weighted avg: 전체 데이터 분포를 반영한 평균 성능을 보고 싶을 때 사용

In [16]:
# 예측값 추출 
pred = model.predict(X_test_ss)

In [17]:
# recall(재현율, 민감도)
recall = recall_score(y_test, pred, average="macro") # macro: 클래스 크기 무시, 균등 가중
recall

1.0

In [18]:
# precision(정밀도)
precision = precision_score(y_test, pred, average="macro")
precision

1.0

In [19]:
# f1 score
f1 = f1_score(y_test, pred, average="macro")
f1

1.0

In [ ]:
# 분류평가지표 
report = classification_report(y_test, pred)
print(report)

# support: 실제 정답 기준 데이터 개수 

              precision    recall  f1-score   support

       Bream       1.00      1.00      1.00         7
       Smelt       1.00      1.00      1.00         3

    accuracy                           1.00        10
   macro avg       1.00      1.00      1.00        10
weighted avg       1.00      1.00      1.00        10



# B) 다중분류 모델 구현 

In [21]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 

from sklearn.model_selection import train_test_split 
from sklearn.preprocessing import StandardScaler 
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import recall_score, precision_score, f1_score, classification_report

## 1. 문제 정의 

- 생선의 특성을 통해 어떤 생선으로 분류되는지 예측하는 이진분류 모델을 구현해보자 

## 2. 데이터 로드 및 탐색 

- 데이터: 생선 데이터
- 포맷: csv 
- feature(특성):
    - Weight: 무게
    - Length: 길이
    - Diagonal: 대각선 길이
    - Height: 높이 
    - width: 두께
- target(정답):
    - Species: 종 (총7종, Bream, Roach, Whitefish, Parkki, Perch, Pike, Smelt)

In [45]:
df = pd.read_csv("https://bit.ly/fish_csv_data")

print(df.shape)
df.head()

(159, 6)


,Species,Weight,Length,Diagonal,Height,Width
0,Bream,242.0,25.4,30.0,11.5200,4.0200
1,Bream,290.0,26.3,31.2,12.4800,4.3056
2,Bream,340.0,26.5,31.1,12.3778,4.6961
3,Bream,363.0,29.0,33.5,12.7300,4.4555
4,Bream,430.0,29.0,34.0,12.4440,5.1340


In [46]:
df.info()

# 데이터 결측치와 자료형 확인
# 결측치: 존재하지 않음 
# 자료형: object 타입 존재 (Species)
# - 수치형: 스케일링 진행 
# - 문자열: 인코딩(수치화) 진행 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 159 entries, 0 to 158
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Species   159 non-null    object 
 1   Weight    159 non-null    float64
 2   Length    159 non-null    float64
 3   Diagonal  159 non-null    float64
 4   Height    159 non-null    float64
 5   Width     159 non-null    float64
dtypes: float64(5), object(1)
memory usage: 7.6+ KB


## 3. 데이터 분리 

### 3.1 feature와 target으로 분리 

In [47]:
X = df.iloc[:, 1:]
y = df[["Species"]]

print(X.shape, y.shape)

(159, 5) (159, 1)


### 3.2 Train, Test 분리 

1. y의 값의 분포 확인 

In [48]:
# target 클래스 분포 확인
y.value_counts()

# 분포가 차이가 날 경우 -> train, test로 분리 시 각 클래스가 동일한 비중으로 분류돼야 함 

Species  
Perch        56
Bream        35
Roach        20
Pike         17
Smelt        14
Parkki       11
Whitefish     6
Name: count, dtype: int64

2. train, test 분리 

In [49]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

(127, 5) (127, 1) (32, 5) (32, 1)


In [50]:
# 클래스 비율 확인 
print(y_train.value_counts(normalize=True))
print()
print(y_test.value_counts(normalize=True))

Species  
Perch        0.354331
Bream        0.220472
Roach        0.125984
Pike         0.102362
Smelt        0.086614
Parkki       0.070866
Whitefish    0.039370
Name: proportion, dtype: float64

Species  
Perch        0.34375
Bream        0.21875
Pike         0.12500
Roach        0.12500
Smelt        0.09375
Parkki       0.06250
Whitefish    0.03125
Name: proportion, dtype: float64


## 4. 전처리: 스케일링 

In [51]:
scaler = StandardScaler()

scaler.fit(X_train)

X_train_ss = scaler.transform(X_train)
X_test_ss = scaler.transform(X_test)

## 5. 모델링 및 정확도 확인 

In [52]:
model = LogisticRegression(C=20, max_iter=1000) # C: default=1, max_iter: default=100

model.fit(X_train_ss, y_train)

# 성능평가(정확도, Accuracy)
print(model.score(X_train_ss, y_train))
print(model.score(X_test_ss, y_test))

0.9291338582677166
0.96875


/opt/anaconda3/envs/sesac-env/lib/python3.11/site-packages/sklearn/utils/validation.py:1352: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [53]:
# 예측값 상위 5개만 확인해보기 

model.predict(X_test_ss[:5])

array(['Bream', 'Roach', 'Smelt', 'Perch', 'Perch'], dtype=object)

In [71]:
# 예측값과 정답 데이터를 매핑하여 결과 비교해보기 

# class명 추출
model_classes = model.classes_ # 모델이 학습한 클래스들의 순서
print("model_classes:")
print(type(model_classes))
print(model_classes)
print("-"*70)

# 클래스에 속할 확률 추출 
proba = model.predict_proba(X_test_ss) # 클래스별 확률 추출 
proba = np.round(proba, 3)
print("proba:")
print(type(proba))
print(proba[0])
print("-"*70)

proba_max_indexes = np.argmax(proba, axis=1) # 각 행의 최댓값 인덱스 확인 
y_test.index = range(len(y_test)) # test 정답지(dataframe)에 index 번호 할당 
print("proba_max_indexes:", proba_max_indexes)
print("y_test.index:", y_test.index)
print("-"*70)

for i, idx in enumerate(proba_max_indexes):
    print(f"{i}번째 데이터")
    print(f"최댓값 인덱스: {idx}, 최댓값: {proba[i][idx]}, 해당 클래스명: {model_classes[idx]}")
    print(f"정답: {y_test.iloc[i, 0]}")
    print()

model_classes:
<class 'numpy.ndarray'>
['Bream' 'Parkki' 'Perch' 'Pike' 'Roach' 'Smelt' 'Whitefish']
----------------------------------------------------------------------
proba:
<class 'numpy.ndarray'>
[0.997 0.    0.    0.    0.    0.    0.002]
----------------------------------------------------------------------
proba_max_indexes: [0 4 5 2 2 2 0 3 5 2 4 4 1 2 3 4 0 2 2 3 2 0 5 4 0 0 1 2 2 2 3 0]
y_test.index: RangeIndex(start=0, stop=32, step=1)
----------------------------------------------------------------------
0번째 데이터
최댓값 인덱스: 0, 최댓값: 0.997, 해당 클래스명: Bream
정답: Bream

1번째 데이터
최댓값 인덱스: 4, 최댓값: 0.703, 해당 클래스명: Roach
정답: Roach

2번째 데이터
최댓값 인덱스: 5, 최댓값: 0.912, 해당 클래스명: Smelt
정답: Smelt

3번째 데이터
최댓값 인덱스: 2, 최댓값: 0.926, 해당 클래스명: Perch
정답: Perch

4번째 데이터
최댓값 인덱스: 2, 최댓값: 0.611, 해당 클래스명: Perch
정답: Perch

5번째 데이터
최댓값 인덱스: 2, 최댓값: 0.78, 해당 클래스명: Perch
정답: Perch

6번째 데이터
최댓값 인덱스: 0, 최댓값: 0.961, 해당 클래스명: Bream
정답: Bream

7번째 데이터
최댓값 인덱스: 3, 최댓값: 0.999, 해당 클래스명: Pike
정답: Pike

8번째 데이터
최댓값 인덱

In [40]:
proba

array([[0.997, 0.   , 0.   , 0.   , 0.   , 0.   , 0.002],
       [0.001, 0.012, 0.206, 0.002, 0.703, 0.   , 0.075],
       [0.   , 0.002, 0.078, 0.   , 0.009, 0.912, 0.   ],
       [0.002, 0.   , 0.926, 0.001, 0.009, 0.   , 0.063],
       [0.   , 0.028, 0.611, 0.001, 0.323, 0.002, 0.035],
       [0.   , 0.003, 0.78 , 0.002, 0.201, 0.002, 0.013],
       [0.961, 0.022, 0.   , 0.   , 0.006, 0.   , 0.01 ],
       [0.   , 0.   , 0.001, 0.999, 0.   , 0.   , 0.   ],
       [0.   , 0.002, 0.04 , 0.   , 0.004, 0.953, 0.   ],
       [0.   , 0.004, 0.736, 0.003, 0.239, 0.001, 0.017],
       [0.015, 0.011, 0.355, 0.009, 0.439, 0.   , 0.171],
       [0.   , 0.061, 0.233, 0.002, 0.644, 0.034, 0.026],
       [0.001, 0.89 , 0.004, 0.   , 0.087, 0.002, 0.015],
       [0.004, 0.   , 0.923, 0.002, 0.   , 0.   , 0.071],
       [0.   , 0.   , 0.002, 0.998, 0.   , 0.   , 0.   ],
       [0.001, 0.039, 0.232, 0.009, 0.673, 0.009, 0.04 ],
       [0.968, 0.02 , 0.   , 0.   , 0.001, 0.   , 0.011],
       [0.001,

In [60]:
y_test.iloc[0]

Species    Bream
Name: 0, dtype: object

## 6. metrics로 모델 평가 

In [65]:
# 예측값 추출 
pred = model.predict(X_test_ss)
pred

array(['Bream', 'Roach', 'Smelt', 'Perch', 'Perch', 'Perch', 'Bream',
       'Pike', 'Smelt', 'Perch', 'Roach', 'Roach', 'Parkki', 'Perch',
       'Pike', 'Roach', 'Bream', 'Perch', 'Perch', 'Pike', 'Perch',
       'Bream', 'Smelt', 'Roach', 'Bream', 'Bream', 'Parkki', 'Perch',
       'Perch', 'Perch', 'Pike', 'Bream'], dtype=object)

In [67]:
# recall: 실제 positive 중 예측도 positive인 비율 

recall = recall_score(y_test, pred, average="macro") # macro: 클래스 크기 무시, 균등 가중 
recall

0.8571428571428571

In [68]:
# precision: positive로 예측한것 중 실제 정답도 positive인 비율
precision = precision_score(y_test, pred, average="macro")
precision

/opt/anaconda3/envs/sesac-env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


0.8285714285714285

In [69]:
# f1score: recall과 precision의 조화평균
# 조화평균을 쓰는 이유: recall과 precision 중 하나라도 낮으면 성능이 나쁘다고 판단해야되기 때문
# 따라서 조화평균을 쓰게되면 작은 값에 더 민감해짐 (한 쪽이 낮으면 전체 값이 극락하며 대신 큰 값에는 영향이 
# 적기 때문에 불균형 데이터에서 특히 중요)

f1 = f1_score(y_test, pred, average="macro") 
f1

0.8412698412698413

In [70]:
report = classification_report(y_test, pred)
print(report)

              precision    recall  f1-score   support

       Bream       1.00      1.00      1.00         7
      Parkki       1.00      1.00      1.00         2
       Perch       1.00      1.00      1.00        11
        Pike       1.00      1.00      1.00         4
       Roach       0.80      1.00      0.89         4
       Smelt       1.00      1.00      1.00         3
   Whitefish       0.00      0.00      0.00         1

    accuracy                           0.97        32
   macro avg       0.83      0.86      0.84        32
weighted avg       0.94      0.97      0.95        32



/opt/anaconda3/envs/sesac-env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/anaconda3/envs/sesac-env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/anaconda3/envs/sesac-env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is

# C) 실습

## 1. 문제 정의 

- 와인의 특성을 통해 레드와인인지 화이트와인인지 예측하는 이진분류 모델을 구현해보자 

In [72]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 

from sklearn.model_selection import train_test_split 
from sklearn.preprocessing import StandardScaler 
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import recall_score, precision_score, f1_score, classification_report

## 2. 데이터 로드 및 탐색 

- 데이터: 와인 데이터
- 포맷: csv 
- feature(특성):
    - alcoho: 알콜 도수 
    - sugar: 당도
    - pH: 산성도
- class: 0.0(레드와인), 1.0(화이트와인)

In [74]:
df = pd.read_csv("https://bit.ly/wine_csv_data")

print(df.shape)
df.head()

(6497, 4)


,alcohol,sugar,pH,class
0,9.4,1.9,3.51,0.0
1,9.8,2.6,3.20,0.0
2,9.8,2.3,3.26,0.0
3,9.8,1.9,3.16,0.0
4,9.4,1.9,3.51,0.0
